In [1]:
# Shiva

# 🧬 Enterprise Repro-Agent: Autonomous Scientific Auditor
### A 5-Day Agents Intensive Capstone Project
**Track:** Enterprise Agents (Productivity & Automation)
**Model:** Gemini 2.5 Flash 


## 🚨 The Problem: The Reproducibility Crisis  
The AI research field faces a **Reproducibility Crisis**.  
Billions of dollars are wasted every year attempting to reproduce results from papers where:

- The published math does **not** match the implementation  
- Hyperparameters differ  
- Key formulas are ambiguous or incorrectly implemented  
- GitHub codebases silently deviate from the PDF  

Researchers waste weeks manually auditing repositories line-by-line to confirm whether a specific equation, algorithmic step, or initialization rule was actually implemented.

Traditional tools like linters only check syntax — **they cannot verify scientific correctness**.

---

## ✅ The Solution: An Autonomous Forensic Auditor  
I built an **Enterprise-Grade Multi-Agent System** powered by **Gemini 2.5 Flash** that automates scientific peer review.

This system is not a chatbot — it is a **Forensic Code Auditor**.

It autonomously:

1. Reads the research paper (PDF/text)
2. Extracts mathematical specifications  
   - algorithms  
   - equations  
   - hyperparameters  
   - training rules  
3. Analyzes the GitHub repository  
4. Hunts through specific files and exact line numbers  
5. Verifies whether the implementation matches the science  
6. Produces a structured reproducibility and deviation report  

It performs the level of analysis normally done by expert reviewers — but faster, more consistently, and with auditable traceability.

---


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bert2019/bert.pdf
/kaggle/input/2017attention/NIPS-2017-attention-is-all-you-need-Paper.pdf
/kaggle/input/agents-intensive-capstone-project/Hackathon dataset.txt


# 🏗️ Enterprise Agent: Setup & Observability
### Features Implemented:
* **Observability:** `logging.getLogger()` is configured to track agent decisions.
* **Dependencies:** Installation of stable `wikipedia` and `pypdf` tools.
* **Authentication:** Secure API key handling using Kaggle Secrets.

In [3]:
# Cell 1: Install Stable Dependencies
!pip install -U -q google-generativeai pypdf wikipedia

import os
import json
import uuid
import logging
import requests
import wikipedia
from datetime import datetime
from pypdf import PdfReader
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

# OBSERVABILITY
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger("EnterpriseSystem")

# AUTH
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    os.environ["GOOGLE_API_KEY"] = api_key
    genai.configure(api_key=api_key)
    print("✅ Authenticated with Gemini 2.5-flash")
except Exception as e:
    print(f"❌ Auth Error: {e}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 21.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but y

In [4]:
import warnings
warnings.filterwarnings('ignore')

# 🛡️ Enterprise Protocols: A2A & Sessions
### Features Implemented:
* **A2A Protocol:** `generate_agent_card` creates standard `agent.json` files for agent discovery.
* **Sessions & Memory:** `InMemorySessionService` implements the official Google ADK pattern for managing conversation history and shared state.
* **MCP Configuration:** `generate_mcp_config` creates a valid config for the Model Context Protocol (GitHub server).

In [5]:
# Cell 2: Official Patterns Implementation

# --- FEATURE: SESSIONS & MEMORY ---
# Implementation of the 'InMemorySessionService' pattern from Google ADK
class InMemorySessionService:
    def __init__(self):
        self._store = {} # The "Database"

    def create_session(self, user_id="default"):
        session_id = f"sess_{uuid.uuid4().hex[:8]}"
        self._store[session_id] = {
            "user_id": user_id,
            "history": [],        # Chat History
            "artifacts": {}       # Context Engineering Artifacts
        }
        logger.info(f"🆕 Session Created: {session_id}")
        return session_id

    def get_history(self, session_id):
        return self._store[session_id]["history"]

    def add_turn(self, session_id, role, text):
        self._store[session_id]["history"].append({"role": role, "parts": [text]})

    def save_artifact(self, session_id, key, data):
        """Save structured context (Context Engineering)"""
        self._store[session_id]["artifacts"][key] = data
        logger.info(f"💾 Artifact Saved: {key} -> Session {session_id}")

    def get_artifact(self, session_id, key):
        return self._store[session_id]["artifacts"].get(key)

# --- FEATURE: A2A PROTOCOL ---
# Implementation of the 'Agent Card' standard for discovery
def generate_agent_card(agent_name, capabilities):
    """Generates an A2A-compliant Agent Card (agent.json)."""
    card = {
        "schema_version": "v1",
        "metadata": {
            "name": agent_name,
            "id": f"agent-{uuid.uuid4().hex[:6]}",
            "created_at": datetime.now().isoformat()
        },
        "capabilities": capabilities,
        "interfaces": {"a2a": {"protocol": "http", "version": "1.0"}}
    }
    # We save this to disk to prove A2A compliance
    filename = f"{agent_name.lower()}_card.json"
    with open(filename, "w") as f:
        json.dump(card, f, indent=2)
    logger.info(f"📇 A2A Card Generated: {filename}")
    return card

# --- FEATURE: MCP CONFIGURATION ---
# We generate the config file required to connect to an MCP server
def generate_mcp_config():
    config = {
        "mcpServers": {
            "github": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-github"]
            }
        }
    }
    with open("mcp_config.json", "w") as f:
        json.dump(config, f, indent=2)
    logger.info("ZC MCP Config Generated: mcp_config.json")

print("✅ Enterprise Protocols (Session, A2A, MCP) Initialized")

✅ Enterprise Protocols (Session, A2A, MCP) Initialized


# 🧠 The Enterprise Agent Class
### Features Implemented:
* **Tools (Custom):** `wiki_search` (Background Check), `inspect_github_content` (Forensic Scanner), `read_pdf_content` (Scientific Extraction).
* **Tools (Built-in):** **Code Execution** is enabled on demand for calculation tasks.
* **Context Engineering:** `inject_artifact` logic allows programmatically injecting structured knowledge into the System Prompt.
* **Model:** Powered by **Gemini 2.5 Flash**.

In [6]:
# Cell 3: Enterprise Agent Class 
import base64
import requests
import json
import wikipedia
from pypdf import PdfReader
from google.generativeai import protos
import google.generativeai as genai

# --- 1. TOOLS ---
def wiki_search(query: str) -> str:
    """Searches Wikipedia to verify the paper's concept."""
    try:
        results = wikipedia.search(query)
        if not results: return "No Wikipedia results found."
        page = wikipedia.page(results[0], auto_suggest=False)
        return f"Wikipedia Title: {page.title}\nSummary: {page.summary[:500]}"
    except Exception as e: return f"Wikipedia Error: {str(e)}"

def inspect_github_content(repo_url: str, specific_file_path: str = None) -> dict:
    """Two-Mode Tool: List files OR Read a specific file."""
    try:
        clean = repo_url.rstrip("/")
        parts = clean.split("/")
        owner, repo = parts[-2], parts[-1]
        if specific_file_path:
            api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{specific_file_path}"
            resp = requests.get(api_url)
            if resp.status_code == 200:
                content = resp.json()['content']
                decoded = base64.b64decode(content).decode('utf-8', errors='ignore')
                return {"file_path": specific_file_path, "code_snippet": decoded[:8000]}
            return {"error": f"Could not read file: {specific_file_path}"}
        api_url = f"https://api.github.com/repos/{owner}/{repo}/contents"
        resp = requests.get(api_url)
        if resp.status_code != 200: return {"error": "Repo not found"}
        files = [i['name'] for i in resp.json()]
        return {"root_files": files}
    except Exception as e: return {"error": str(e)}

def read_pdf_content(pdf_path: str) -> str:
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for i, page in enumerate(reader.pages):
            if i >= 12: break
            text += page.extract_text() + "\n"
        return text
    except Exception as e: return f"Error: {str(e)}"

# --- 2. AGENT CLASS ---
class EnterpriseAgent:
    def __init__(self, name, instructions, tools=None, enable_code_execution=False):
        self.name = name
        self.instructions = instructions
        self.session_service = InMemorySessionService() 
        self.active_tools = tools if tools else []
        if enable_code_execution:
            self.active_tools.append("code_execution")
        self.model = genai.GenerativeModel(
            model_name="gemini-2.5-flash", 
            tools=self.active_tools, 
            system_instruction=instructions
        )
        generate_agent_card(name, ["reasoning", "forensic-analysis"])

    def run(self, session_id, prompt, inject_artifact=None):
        final_prompt = prompt
        if inject_artifact:
            context_str = json.dumps(inject_artifact, indent=2)
            final_prompt = f"""[SCIENTIFIC TRUTH FROM PAPER]\n{context_str}\n\n[YOUR TASK]\n{prompt}"""

        history = self.session_service.get_history(session_id)
        chat = self.model.start_chat(history=history, enable_automatic_function_calling=True)
        
        logger.info(f"🤖 {self.name} investigating...")
        try:
            # 🛠️ FIX: Explicitly handle code execution response
            response = chat.send_message(final_prompt)
            self.session_service.add_turn(session_id, "user", final_prompt)
            
            # Check for text FIRST, even if code was executed
            if response.text:
                self.session_service.add_turn(session_id, "model", response.text)
                return response.text
            
            # Fallback for code execution without text output
            if response.parts and response.parts[0].executable_code:
                 # Try to get the next turn which might contain the text report
                try:
                    next_response = chat.send_message("Please provide the final report text based on the code execution.")
                    if next_response.text:
                         self.session_service.add_turn(session_id, "model", next_response.text)
                         return next_response.text
                except:
                    pass # Fallback if that fails
                return "Code Executed Successfully, but no final text report was generated."
                
            return "Task Completed."
        except Exception as e:
            return f"Execution Error: {e}"

print("✅ Enterprise Forensic Agents Ready ")

✅ Enterprise Forensic Agents Ready 


# 🚀 Execution: The Forensic Math Audit
### Workflow:
1. **Phase 1 (Search):** Verify the paper exists and identify key concepts (using Wikipedia).
2. **Phase 2 (Extraction):** Read the PDF to extract specific mathematical claims (e.g., Activation Functions).
3. **Phase 3 (Forensics):** Inject the paper's claims into the Code Auditor. The agent then:
    * Lists repo files.
    * Identifies `modeling.py`.
    * **Reads the code** to find the exact math implementation.
4. **Phase 4 (Reporting):** Generate a "Discrepancy Report" with a score and diff table.

In [7]:
# Cell 4: Run the Forensic Math Audit
from IPython.display import HTML
import markdown
import os

# 1. Setup Services
global_session_service = InMemorySessionService()
session_id = global_session_service.create_session(user_id="kaggle_judge")
generate_mcp_config()

# 2. Define Agents
bg_checker = EnterpriseAgent(
    "Background_Checker", 
    "Verify paper credibility via Wikipedia.", 
    tools=[wiki_search]
)
bg_checker.session_service = global_session_service 

paper_analyst = EnterpriseAgent(
    "Paper_Analyst", 
    "Extract the 'Key Algorithms' and 'Mathematical Formulas' from PDF.", 
    tools=[read_pdf_content]
)
paper_analyst.session_service = global_session_service

code_forensics = EnterpriseAgent(
    "Code_Forensics", 
    "You are a Mathematical Code Auditor. Your job is to find the ACTUAL PYTHON FUNCTION DEFINITION (def ...) and extract the math logic.", 
    tools=[inspect_github_content] 
)
code_forensics.session_service = global_session_service

# Academic Report Writer
writer = EnterpriseAgent(
    "Report_Writer", 
    "You are a Scientific Reviewer. Write a formal, academic-style report on the reproducibility of the paper's claims in the codebase. Use formal language, clear headings (Abstract, Methodology, Results, Discussion), and cite the provided evidence.", 
    enable_code_execution=True
)
writer.session_service = global_session_service

# --- EXECUTION ---
pdf_path = "/kaggle/input/bert2019/bert.pdf" 
repo_url = "https://github.com/google-research/bert"

print(f"\n🚀 Starting Forensic Math Audit: {session_id}\n")

# PHASE 1: Verify Topic
print("🌍 Phase 1: Topic Verification")
bg_info = bg_checker.run(session_id, f"Search for 'BERT language model'")

# PHASE 2: Extract Algorithms
print("📝 Phase 2: Extracting Math Specs")
paper_math = paper_analyst.run(session_id, f"Read {pdf_path}. explicitly extract the Activation Function (Gelu/Relu) and Optimizer details.")

research_artifact = {"source": "BERT", "algorithms": paper_math[:1000]}
global_session_service.save_artifact(session_id, "research_math", research_artifact)
print(f"   > Specs Captured: {paper_math[:100]}...\n")

# PHASE 3: Forensic Code Audit
print("💻 Phase 3: Forensic Code Audit (Hunting for Math)")
audit_findings = code_forensics.run(
    session_id, 
    f"""
    Target Repo: {repo_url}
    
    TASK 1: Find the file 'modeling.py'.
    TASK 2: Read it and find the function definition `def gelu` or `def get_activation`.
    TASK 3: Extract the EXACT 3-4 lines of code that perform the mathematical calculation.
    
    Compare this math against the standard definition.
    """,
    inject_artifact=research_artifact 
)
print(f"   > Forensics Complete.\n")

# PHASE 4: Final Academic Report
print("⚖️ Phase 4: Final Academic Forensic Report")
final_prompt = f"""
[EVIDENCE 1: PAPER SPECS] {paper_math}
[EVIDENCE 2: CODE IMPLEMENTATION] {audit_findings}

Generate a formal **Reproducibility Audit Report** in an academic style.

**Structure:**
1.  **Abstract:** A brief summary.
2.  **Methodology:** Describe the audit process.
3.  **Results:**
    * **Reproducibility Score:** Assign a score (1-4) with a justification.
    * **Mathematical Verification:** Present the code snippet and analyze its equivalence.
    * **Discrepancy Analysis:** Provide a table comparing the "Paper Claim" vs. "Code Reality".
4.  **Discussion:** Discuss implications and ambiguity.
5.  **Conclusion:** A final statement.
6.  **References:** List sources.

**Tone:** Formal, objective, and scientific.
"""
final_report = writer.run(session_id, final_prompt)

# 🛠️ UPDATE: Use HTML display to prevent truncation
html_report = markdown.markdown(final_report, extensions=['tables'])
display(HTML(f"<h1>📜 Academic Forensic Audit Report</h1>{html_report}"))

# Kaggle-Friendly File Saving
output_dir = "/kaggle/working/"
os.makedirs(output_dir, exist_ok=True)
report_filename = os.path.join(output_dir, f"submission_report.md")
with open(report_filename, "w") as f:
    f.write(final_report)

print(f"\n✅ Report Saved to Disk: {report_filename}")
print("   (This file will be available in the 'Output' tab after the notebook finishes running.)")

2025-11-25 21:11:34,800 | INFO | 🆕 Session Created: sess_ee47b2ea
2025-11-25 21:11:34,801 | INFO | ZC MCP Config Generated: mcp_config.json
2025-11-25 21:11:35,022 | INFO | 📇 A2A Card Generated: background_checker_card.json
2025-11-25 21:11:35,025 | INFO | 📇 A2A Card Generated: paper_analyst_card.json
2025-11-25 21:11:35,028 | INFO | 📇 A2A Card Generated: code_forensics_card.json
2025-11-25 21:11:35,030 | INFO | 📇 A2A Card Generated: report_writer_card.json
2025-11-25 21:11:35,031 | INFO | 🤖 Background_Checker investigating...



🚀 Starting Forensic Math Audit: sess_ee47b2ea

🌍 Phase 1: Topic Verification


2025-11-25 21:11:41,208 | INFO | 🤖 Paper_Analyst investigating...


📝 Phase 2: Extracting Math Specs


2025-11-25 21:11:46,160 | INFO | 💾 Artifact Saved: research_math -> Session sess_ee47b2ea
2025-11-25 21:11:46,162 | INFO | 🤖 Code_Forensics investigating...


   > Specs Captured: The document states the following:

**Activation Function:**
The paper references "Gaussian Error Li...

💻 Phase 3: Forensic Code Audit (Hunting for Math)


2025-11-25 21:11:51,624 | INFO | 🤖 Report_Writer investigating...


   > Forensics Complete.

⚖️ Phase 4: Final Academic Forensic Report


Feature,Paper Claim (EVIDENCE 1),Code Reality (EVIDENCE 2),Reproducibility Status
Activation Function,"""BERT uses GELU, which is a smoother approximation of ReLU."" References ""Gaussian Error Linear Units (GELU)"" from Hendrycks and Gimpel (2016).","def gelu(x): ... cdf = 0.5 * (1.0 + tf.tanh((np.sqrt(2 / np.pi) * (x + 0.044715 * tf.pow(x, 3))))) return x * cdf. This implements the tanh approximation of GELU.",Verified
Optimizer Details,"Learning rate (5e-5, 4e-5, 3e-5, 2e-5), batch sizes (32 for GLUE/SQuAD v1.1, 48 for SQuAD v2.0). Exact optimizer (e.g., Adam, SGD) not explicitly named.",Not applicable for direct comparison of the specific optimizer algorithm as it was not explicitly named in the paper claim. Parameters like learning rates and batch sizes would be configured separately from the optimizer's core mathematical definition.,Undetermined*



✅ Report Saved to Disk: /kaggle/working/submission_report.md
   (This file will be available in the 'Output' tab after the notebook finishes running.)


In [8]:
import google.generativeai as genai
from google.generativeai import generative_models

print("🔍 Gemini SDK Version:", genai.__version__)
print("\n=== Testing Google Search Tool Support ===\n")

tests = {}

# Test 1 — tools=[{"google_search":{}}]
try:
    genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        tools=[{"google_search": {}}]
    )
    tests["tools=[{'google_search':{}}]"] = "✔ WORKS"
except Exception as e:
    tests["tools=[{'google_search':{}}]"] = f"❌ {type(e).__name__}: {e}"

# Test 2 — tools=[{"google_search_retrieval":{}}]
try:
    genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        tools=[{"google_search_retrieval": {}}]
    )
    tests["tools=[{'google_search_retrieval':{}}]"] = "✔ WORKS"
except Exception as e:
    tests["tools=[{'google_search_retrieval':{}}]"] = f"❌ {type(e).__name__}: {e}"

# Test 3 — tool_config={"google_search_retrieval":{}}
try:
    genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        tool_config={"google_search_retrieval": {}}
    )
    tests["tool_config={'google_search_retrieval':{}}"] = "✔ WORKS"
except Exception as e:
    tests["tool_config={'google_search_retrieval':{}}"] = f"❌ {type(e).__name__}: {e}"

# Test 4 — ToolType: GoogleSearchRetrieval()
try:
    from google.generativeai.types import tool_types
    search_tool = tool_types.GoogleSearchRetrieval()
    genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        tools=[search_tool]
    )
    tests["tool_types.GoogleSearchRetrieval()"] = "✔ WORKS"
except Exception as e:
    tests["tool_types.GoogleSearchRetrieval()"] = f"❌ {type(e).__name__}: {e}"

# Show results
print("=== RESULTS ===")
for name, result in tests.items():
    print(f"{name:45}  →  {result}")


🔍 Gemini SDK Version: 0.8.5

=== Testing Google Search Tool Support ===

=== RESULTS ===
tools=[{'google_search':{}}]                   →  ❌ ValueError: Unknown field for FunctionDeclaration: google_search
tools=[{'google_search_retrieval':{}}]         →  ✔ WORKS
tool_config={'google_search_retrieval':{}}     →  ❌ KeyError: 'function_calling_config'
tool_types.GoogleSearchRetrieval()             →  ❌ ImportError: cannot import name 'tool_types' from 'google.generativeai.types' (/usr/local/lib/python3.11/dist-packages/google/generativeai/types/__init__.py)


In [9]:
import google.generativeai as genai

print("SDK Version:", genai.__version__)

print("\n=== AVAILABLE MODELS (v1beta) ===")
models = genai.list_models()

for m in models:
    print("-", m.name)


SDK Version: 0.8.5

=== AVAILABLE MODELS (v1beta) ===
- models/embedding-gecko-001
- models/gemini-2.5-pro-preview-03-25
- models/gemini-2.5-flash
- models/gemini-2.5-pro-preview-05-06
- models/gemini-2.5-pro-preview-06-05
- models/gemini-2.5-pro
- models/gemini-2.0-flash-exp
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-exp-image-generation
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.0-flash-lite-preview-02-05
- models/gemini-2.0-flash-lite-preview
- models/gemini-2.0-pro-exp
- models/gemini-2.0-pro-exp-02-05
- models/gemini-exp-1206
- models/gemini-2.0-flash-thinking-exp-01-21
- models/gemini-2.0-flash-thinking-exp
- models/gemini-2.0-flash-thinking-exp-1219
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/learnlm-2.0-flash-experimental
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-

In [10]:
# # Cell 3: Enterprise Agent Class (Forensic Code Tools)
# import base64
# import requests
# import json
# import wikipedia
# from pypdf import PdfReader
# from google.generativeai import protos
# import google.generativeai as genai

# # --- 1. TOOLS ---

# def wiki_search(query: str) -> str:
#     """Searches Wikipedia to verify the paper's concept."""
#     try:
#         # Broader search to ensure hits
#         results = wikipedia.search(query)
#         if not results: return "No Wikipedia results found."
#         # Get the most relevant page
#         page = wikipedia.page(results[0], auto_suggest=False)
#         return f"Wikipedia Title: {page.title}\nSummary: {page.summary[:500]}"
#     except Exception as e: return f"Wikipedia Error: {str(e)}"

# def inspect_github_content(repo_url: str, specific_file_path: str = None) -> dict:
#     """
#     Two-Mode Tool:
#     1. If 'specific_file_path' is None -> Lists all files in the repo root.
#     2. If 'specific_file_path' is provided (e.g. 'modeling.py') -> Reads the ACTUAL CODE.
#     """
#     try:
#         clean = repo_url.rstrip("/")
#         parts = clean.split("/")
#         owner, repo = parts[-2], parts[-1]
        
#         # Mode 2: Read Specific File (Forensic Mode)
#         if specific_file_path:
#             api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{specific_file_path}"
#             resp = requests.get(api_url)
#             if resp.status_code == 200:
#                 content = resp.json()['content']
#                 decoded = base64.b64decode(content).decode('utf-8', errors='ignore')
#                 # Return first 8000 chars of code (enough to check algorithms)
#                 return {"file_path": specific_file_path, "code_snippet": decoded[:8000]}
#             return {"error": f"Could not read file: {specific_file_path}"}
            
#         # Mode 1: List Files (Discovery Mode)
#         api_url = f"https://api.github.com/repos/{owner}/{repo}/contents"
#         resp = requests.get(api_url)
#         if resp.status_code != 200: return {"error": "Repo not found"}
#         files = [i['name'] for i in resp.json()]
#         return {"root_files": files}

#     except Exception as e: return {"error": str(e)}

# def read_pdf_content(pdf_path: str) -> str:
#     try:
#         reader = PdfReader(pdf_path)
#         text = ""
#         for i, page in enumerate(reader.pages):
#             if i >= 12: break # Read more pages for algorithm details
#             text += page.extract_text() + "\n"
#         return text
#     except Exception as e: return f"Error: {str(e)}"

# # --- 2. AGENT CLASS ---
# class EnterpriseAgent:
#     def __init__(self, name, instructions, tools=None, enable_code_execution=False):
#         self.name = name
#         self.instructions = instructions
#         self.session_service = InMemorySessionService() 
#         self.active_tools = tools if tools else []
        
#         if enable_code_execution:
#             self.active_tools.append("code_execution")
        
#         # Using Gemini 2.5 Flash for Deep Reasoning
#         self.model = genai.GenerativeModel(
#             model_name="gemini-2.5-flash", 
#             tools=self.active_tools, 
#             system_instruction=instructions
#         )
#         generate_agent_card(name, ["reasoning", "forensic-analysis"])

#     def run(self, session_id, prompt, inject_artifact=None):
#         final_prompt = prompt
        
#         if inject_artifact:
#             # We inject the "Scientific Truth" to compare against
#             context_str = json.dumps(inject_artifact, indent=2)
#             final_prompt = f"""
#             [SCIENTIFIC TRUTH FROM PAPER]
#             {context_str}
            
#             [YOUR TASK]
#             {prompt}
#             """

#         history = self.session_service.get_history(session_id)
#         chat = self.model.start_chat(history=history, enable_automatic_function_calling=True)
        
#         logger.info(f"🤖 {self.name} investigating...")
#         try:
#             response = chat.send_message(final_prompt)
#             self.session_service.add_turn(session_id, "user", final_prompt)
            
#             if response.parts and response.parts[0].text:
#                 self.session_service.add_turn(session_id, "model", response.text)
#                 return response.text
            
#             if response.parts and response.parts[0].executable_code:
#                 return "Code Executed Successfully."
                
#             return "Task Completed."
#         except Exception as e:
#             return f"Execution Error: {e}"

# print("✅ Enterprise Forensic Agents Ready")

In [11]:
# # # Cell 4: Run the Forensic Math Audit
# # from IPython.display import Markdown

# # # 1. Setup Services
# # global_session_service = InMemorySessionService()
# # session_id = global_session_service.create_session(user_id="kaggle_judge")
# # generate_mcp_config()

# # # 2. Define Agents
# # bg_checker = EnterpriseAgent(
# #     "Background_Checker", 
# #     "Verify paper credibility via Wikipedia.", 
# #     tools=[wiki_search]
# # )
# # bg_checker.session_service = global_session_service 

# # paper_analyst = EnterpriseAgent(
# #     "Paper_Analyst", 
# #     "Extract the 'Key Algorithms' and 'Mathematical Formulas' from PDF.", 
# #     tools=[read_pdf_content]
# # )
# # paper_analyst.session_service = global_session_service

# # code_forensics = EnterpriseAgent(
# #     "Code_Forensics", 
# #     "You are a Mathematical Code Auditor. Your job is to find the ACTUAL PYTHON FUNCTION DEFINITION (def ...) and extract the math logic.", 
# #     tools=[inspect_github_content] 
# # )
# # code_forensics.session_service = global_session_service

# # writer = EnterpriseAgent("Report_Writer", "Write a Forensic Audit Report.", enable_code_execution=True)
# # writer.session_service = global_session_service

# # # --- EXECUTION ---
# # pdf_path = "/kaggle/input/bert2019/bert.pdf" 
# # repo_url = "https://github.com/google-research/bert"

# # print(f"\n🚀 Starting Forensic Math Audit: {session_id}\n")

# # # PHASE 1: Verify Topic
# # print("🌍 Phase 1: Topic Verification")
# # bg_info = bg_checker.run(session_id, f"Search for 'BERT language model'")

# # # PHASE 2: Extract Algorithms
# # print("📝 Phase 2: Extracting Math Specs")
# # paper_math = paper_analyst.run(session_id, f"Read {pdf_path}. explicitly extract the Activation Function (Gelu/Relu) and Optimizer details.")

# # research_artifact = {"source": "BERT", "algorithms": paper_math[:1000]}
# # global_session_service.save_artifact(session_id, "research_math", research_artifact)
# # print(f"   > Specs Captured: {paper_math[:100]}...\n")

# # # PHASE 3: Forensic Code Audit (Aggressive Math Hunt)
# # print("💻 Phase 3: Forensic Code Audit (Hunting for Math)")
# # audit_findings = code_forensics.run(
# #     session_id, 
# #     f"""
# #     Target Repo: {repo_url}
    
# #     TASK 1: Find the file 'modeling.py'.
# #     TASK 2: Read it and find the function definition `def gelu` or `def get_activation`.
# #     TASK 3: Extract the EXACT 3-4 lines of code that perform the mathematical calculation.
    
# #     Compare this math against the standard definition.
# #     """,
# #     inject_artifact=research_artifact 
# # )
# # print(f"   > Forensics Complete.\n")

# # # PHASE 4: Technical Report (Score & Diff)
# # print("⚖️ Phase 4: Final Forensic Report")
# # final_prompt = f"""
# # [EVIDENCE 1: PAPER SPECS] {paper_math}
# # [EVIDENCE 2: CODE IMPLEMENTATION] {audit_findings}

# # Generate a "Reproducibility Discrepancy Report".

# # 1. **Reproducibility Score (1-4):**
# #    - 4: Perfect match.
# #    - 3: Match, but paper was vague (Ambiguity Penalty).
# #    - 2: Minor code deviations.
# #    - 1: Major mismatch.

# # 2. **The "Math Check":**
# #    - Show the Code Snippet found.
# #    - Explain if the formula is mathematically equivalent to the paper's claim.

# # 3. **Discrepancy Table:**
# #    - Create a Markdown Table comparing "Paper Claim" vs "Code Reality".
   
# # 4. **Implementation Proof:**
# #    - Quote the exact file name and line number range found.
# # """
# # final_report = writer.run(session_id, final_prompt)

# # display(Markdown(f"# 📊 Forensic Audit Report\n\n{final_report}"))
# # # FEATURE: Save Report to Disk (Enterprise Requirement)
# # report_filename = f"reproducibility_report_{session_id}.md"
# # with open(report_filename, "w") as f:
# #     f.write(final_report)

# # print(f"\n✅ Report Saved to Disk: {report_filename}")
# # print("   (You can download this file from the 'Output' tab)")
# # Cell 4: Run the Forensic Math Audit
# from IPython.display import Markdown
# import os

# # 1. Setup Services
# global_session_service = InMemorySessionService()
# session_id = global_session_service.create_session(user_id="kaggle_judge")
# generate_mcp_config()

# # 2. Define Agents
# bg_checker = EnterpriseAgent(
#     "Background_Checker", 
#     "Verify paper credibility via Wikipedia.", 
#     tools=[wiki_search]
# )
# bg_checker.session_service = global_session_service 

# paper_analyst = EnterpriseAgent(
#     "Paper_Analyst", 
#     "Extract the 'Key Algorithms' and 'Mathematical Formulas' from PDF.", 
#     tools=[read_pdf_content]
# )
# paper_analyst.session_service = global_session_service

# code_forensics = EnterpriseAgent(
#     "Code_Forensics", 
#     "You are a Mathematical Code Auditor. Your job is to find the ACTUAL PYTHON FUNCTION DEFINITION (def ...) and extract the math logic.", 
#     tools=[inspect_github_content] 
# )
# code_forensics.session_service = global_session_service

# # 🛠️ UPDATE: Academic Report Writer Instructions
# writer = EnterpriseAgent(
#     "Report_Writer", 
#     "You are a Scientific Reviewer. Write a formal, academic-style report on the reproducibility of the paper's claims in the codebase. Use formal language, clear headings (Abstract, Methodology, Results, Discussion), and cite the provided evidence.", 
#     enable_code_execution=True
# )
# writer.session_service = global_session_service

# # --- EXECUTION ---
# pdf_path = "/kaggle/input/bert2019/bert.pdf" 
# repo_url = "https://github.com/google-research/bert"

# print(f"\n🚀 Starting Forensic Math Audit: {session_id}\n")

# # PHASE 1: Verify Topic
# print("🌍 Phase 1: Topic Verification")
# bg_info = bg_checker.run(session_id, f"Search for 'BERT language model'")

# # PHASE 2: Extract Algorithms
# print("📝 Phase 2: Extracting Math Specs")
# paper_math = paper_analyst.run(session_id, f"Read {pdf_path}. explicitly extract the Activation Function (Gelu/Relu) and Optimizer details.")

# research_artifact = {"source": "BERT", "algorithms": paper_math[:1000]}
# global_session_service.save_artifact(session_id, "research_math", research_artifact)
# print(f"   > Specs Captured: {paper_math[:100]}...\n")

# # PHASE 3: Forensic Code Audit (Aggressive Math Hunt)
# print("💻 Phase 3: Forensic Code Audit (Hunting for Math)")
# audit_findings = code_forensics.run(
#     session_id, 
#     f"""
#     Target Repo: {repo_url}
    
#     TASK 1: Find the file 'modeling.py'.
#     TASK 2: Read it and find the function definition `def gelu` or `def get_activation`.
#     TASK 3: Extract the EXACT 3-4 lines of code that perform the mathematical calculation.
    
#     Compare this math against the standard definition.
#     """,
#     inject_artifact=research_artifact 
# )
# print(f"   > Forensics Complete.\n")

# # PHASE 4: Technical Report (Score & Diff)
# print("⚖️ Phase 4: Final Academic Forensic Report")
# final_prompt = f"""
# [EVIDENCE 1: PAPER SPECS] {paper_math}
# [EVIDENCE 2: CODE IMPLEMENTATION] {audit_findings}

# Generate a formal **Reproducibility Audit Report** in an academic style.

# **Structure:**
# 1.  **Abstract:** A brief summary of the audit's purpose, methods, and main finding (the reproducibility score).
# 2.  **Methodology:** Describe the process of auditing the paper against the codebase.
# 3.  **Results:**
#     * **Reproducibility Score:** Assign a score (1-4) with a justification.
#     * **Mathematical Verification:** Present the identified code snippet for the activation function and analyze its mathematical equivalence to the paper's claim.
#     * **Discrepancy Analysis:** Provide a table comparing the "Paper Claim" vs. "Code Reality".
# 4.  **Discussion:** Discuss the implications of the findings, including any ambiguity in the paper and the correctness of the implementation.
# 5.  **Conclusion:** A final concluding statement on the reproducibility of the audited feature.
# 6.  **References:** List the paper and repository as sources.

# **Tone:** Formal, objective, and scientific.
# """
# final_report = writer.run(session_id, final_prompt)

# display(Markdown(f"# 📜 Academic Forensic Audit Report\n\n{final_report}"))

# # 🛠️ UPDATE: Kaggle-Friendly File Saving
# # Save to /kaggle/working/ for persistence after commit
# output_dir = "/kaggle/working/"
# os.makedirs(output_dir, exist_ok=True) # Ensure directory exists
# report_filename = os.path.join(output_dir, f"submission_report.md")

# with open(report_filename, "w") as f:
#     f.write(final_report)

# print(f"\n✅ Report Saved to Disk: {report_filename}")
# print("   (This file will be available in the 'Output' tab after the notebook finishes running.)")

# # Optional: Read back the report to confirm it was saved correctly
# # with open(report_filename, "r") as f:
# #     print("\n--- Verifying Saved Report Content ---")
# #     print(f.read()[:500] + "...\n(truncated)")

In [12]:
# # Cell 3: Enterprise Agent Class (Wikipedia + Code Execution)
# import base64
# import requests
# import json
# from pypdf import PdfReader
# from google.generativeai import protos
# # --- 1. CUSTOM TOOL: Wikipedia Search ---
# def wiki_search(query: str) -> str:
#     """Searches Wikipedia to verify if a topic is real and popular."""
#     try:
#         # Get summary of first result
#         results = wikipedia.search(query)
#         if not results: return "No Wikipedia results found."
#         summary = wikipedia.summary(results[0], sentences=3)
#         return f"Wikipedia Summary for '{results[0]}': {summary}"
#     except Exception as e: return f"Wikipedia Error: {str(e)}"

# # --- 2. SMART TOOL: GitHub Scanner + README Reader ---
# def check_github_files(repo_url: str) -> dict:
#     """Scans a repo AND reads the README.md for installation details."""
#     try:
#         clean = repo_url.rstrip("/")
#         parts = clean.split("/")
#         owner, repo = parts[-2], parts[-1]
        
#         # A. Get File Structure
#         api_url = f"https://api.github.com/repos/{owner}/{repo}/contents"
#         resp = requests.get(api_url)
#         if resp.status_code == 403: return {"error": "GitHub API Rate Limit Exceeded. Try again in 1 hour."}
#         if resp.status_code != 200: return {"error": "Repo not found"}
        
#         data = resp.json()
#         files = [i['name'] for i in data]
        
#         # B. Fetch README Content (The Upgrade)
#         readme_text = "No README found."
#         if "README.md" in files:
#             readme_url = f"https://api.github.com/repos/{owner}/{repo}/contents/README.md"
#             r_resp = requests.get(readme_url)
#             if r_resp.status_code == 200:
#                 # GitHub sends content as Base64
#                 content = r_resp.json()['content']
#                 decoded = base64.b64decode(content).decode('utf-8', errors='ignore')
#                 readme_text = decoded[:4000] # Limit to 4000 chars to save context
        
#         return {
#             "structure": files,
#             "readme_snippet": readme_text
#         }
#     except Exception as e: return {"error": str(e)}

# def read_pdf_content(pdf_path: str) -> str:
#     """Reads text from a local PDF."""
#     try:
#         reader = PdfReader(pdf_path)
#         text = ""
#         for i, page in enumerate(reader.pages):
#             if i >= 5: break
#             text += page.extract_text() + "\n"
#         return text
#     except Exception as e: return f"Error: {str(e)}"

# # --- 3. AGENT CLASS ---
# class EnterpriseAgent:
#     def __init__(self, name, instructions, tools=None, enable_code_execution=False):
#         self.name = name
#         self.instructions = instructions
#         self.session_service = InMemorySessionService() 
        
#         # Base Tools (Custom)
#         self.active_tools = tools if tools else []
        
#         # 🛠️ FEATURE: Built-in Code Execution (Official Tool)
#         # We pass 'code_execution' as a string which the SDK maps to the official sandbox
#         if enable_code_execution:
#             self.active_tools.append("code_execution")
        
#         # CONFIG: Gemini 2.5 Flash
#         self.model = genai.GenerativeModel(
#             model_name="gemini-2.5-flash", 
#             tools=self.active_tools, 
#             system_instruction=instructions
#         )
        
#         generate_agent_card(name, ["reasoning", "tool-use"])

#     def run(self, session_id, prompt, inject_artifact=None):
#         # Context Engineering
#         if inject_artifact:
#             logger.info(f"💉 Injecting Context into {self.name}...")
#             dynamic_system_prompt = f"""
#             {self.instructions}
#             [CRITICAL CONTEXT]
#             {json.dumps(inject_artifact, indent=2)}
#             """
#             self.model._system_instruction = dynamic_system_prompt

#         history = self.session_service.get_history(session_id)
#         # Note: Code Execution requires function calling enabled
#         chat = self.model.start_chat(history=history, enable_automatic_function_calling=True)
        
#         logger.info(f"🤖 {self.name} executing...")
#         try:
#             response = chat.send_message(prompt)
#             self.session_service.add_turn(session_id, "user", prompt)
            
#             # Handle Text Response
#             if response.parts and response.parts[0].text:
#                 self.session_service.add_turn(session_id, "model", response.text)
#                 return response.text
            
#             # Handle Code Execution Response (Executable Code)
#             if response.parts and response.parts[0].executable_code:
#                 return "Code Executed Successfully inside Sandbox."
                
#             return "Task Completed."
                
#         except Exception as e:
#             return f"Execution Error: {e}"

# print("✅ Enterprise Agent Class Ready (Wikipedia + Code Execution)")

In [13]:
# # Cell 4: Run the Robust Pipeline (Fixed for Context Passing)
# from IPython.display import Markdown

# # 1. Setup Services
# global_session_service = InMemorySessionService()
# session_id = global_session_service.create_session(user_id="kaggle_judge")
# generate_mcp_config()

# # 2. Define Agents
# bg_checker = EnterpriseAgent(
#     name="Background_Checker",
#     instructions="You are a Librarian. Verify paper credibility.",
#     tools=[wiki_search]
# )
# bg_checker.session_service = global_session_service 

# paper_analyst = EnterpriseAgent(
#     name="Paper_Analyst",
#     instructions="Extract 'Ideal Setup' from PDF. Output the raw text of what you found.",
#     tools=[read_pdf_content] 
# )
# paper_analyst.session_service = global_session_service

# code_auditor = EnterpriseAgent(
#     name="Code_Auditor",
#     instructions="Verify code files. Output a list of MISSING files and MATCHING files.",
#     tools=[check_github_files]
# )
# code_auditor.session_service = global_session_service

# writer = EnterpriseAgent(
#     name="Report_Writer", 
#     instructions="You are a Judge. Use the provided Context to write a final score and report.",
#     enable_code_execution=True 
# )
# writer.session_service = global_session_service

# # --- PIPELINE ---
# pdf_path = "/kaggle/input/2017attention/NIPS-2017-attention-is-all-you-need-Paper.pdf"
# repo_url = "https://github.com/tensorflow/tensor2tensor"

# print(f"\n🚀 Starting Robust Session: {session_id}\n")

# # PHASE 1: Background Check
# print("🌍 Phase 1: Search Verification")
# bg_info = bg_checker.run(session_id, f"Search for 'Attention Is All You Need' paper.")
# print(f"   > Background: {bg_info[:100]}...\n")

# # PHASE 2: Paper Analysis
# print("📝 Phase 2: Analyzing PDF")
# # We explicitly ask for a detailed summary to pass forward
# paper_findings = paper_analyst.run(session_id, f"Read {pdf_path}. List the Dataset, Model, and Hyperparameters found.")

# # Create Artifact (Context Engineering)
# research_artifact = {
#     "source": "Attention Is All You Need",
#     "findings": paper_findings[:1000] # Capture the text!
# }
# global_session_service.save_artifact(session_id, "research_specs", research_artifact)
# print(f"   > Paper Findings Captured.\n")

# # PHASE 3: Code Audit
# print("💻 Phase 3: Code Audit")
# audit_findings = code_auditor.run(
#     session_id, 
#     f"Check this repo: {repo_url}. Compare against these findings: {paper_findings}",
#     inject_artifact=research_artifact 
# )
# print(f"   > Audit Findings Captured.\n")

# # PHASE 4: Final Report (CRITICAL FIX HERE)
# print("⚖️ Phase 4: Final Report")

# # We construct a Mega-Prompt containing all previous evidence
# final_prompt = f"""
# GENERATE FINAL REPRODUCIBILITY REPORT.

# [EVIDENCE 1: BACKGROUND]
# {bg_info}

# [EVIDENCE 2: PAPER SPECS]
# {paper_findings}

# [EVIDENCE 3: CODE AUDIT RESULTS]
# {audit_findings}

# TASK:
# 1. Assign a Reproducibility Score (1-4).
# 2. List 3 strengths and 3 weaknesses.
# 3. Use Python to calculate the score if needed.
# """

# final_report = writer.run(session_id, final_prompt)

# display(Markdown(f"# 📊 Robust Reproducibility Report\n\n{final_report}"))